In [1]:
import pandas as pd


df = pd.read_csv('/kaggle/input/gymshark-products-dataset/gymshark_products.csv')


print("=== INITIAL DATA EXPLORATION ===")
print("Dataset Info:")
df.info()
print(f"\nDataset Shape: {df.shape}")
print(f"\nMissing Values:\n{df.isnull().sum()}")


print("\n\n=== DATA CLEANING PROCESS ===")


df_clean = df.drop("inventory_quantity", axis=1)
df_clean = df_clean[df_clean['price'] > 0]

print(f"After initial cleaning: {len(df_clean)} products")


classification_rules = {
    'beanie': 'Accessories - Headwear',
    'cap': 'Accessories - Headwear',
    'sock': 'Accessories - Footwear',
    'bag': 'Accessories - Bags'
}

def quick_classify(title, tags):
    text = (title + ' ' + tags).lower()
    for keyword, category in classification_rules.items():
        if keyword in text:
            return category
    return 'Accessories'


missing_mask = df_clean['product_type'].isnull()
df_clean.loc[missing_mask, 'product_type'] = df_clean[missing_mask].apply(
    lambda row: quick_classify(row['title'], row['tags']), axis=1
)

print(f"Products without product_type after classification: {df_clean['product_type'].isnull().sum()}")


def greedy_image_filler(df):
    missing_mask = df['image_src'].isnull()
    filled_count = 0
    
    for idx in df[missing_mask].index:
        current_product = df.loc[idx]
        
        
        same_handle = df[(df['handle'] == current_product['handle']) & 
                        (df['image_src'].notnull())]
        if len(same_handle) > 0:
            df.loc[idx, 'image_src'] = same_handle['image_src'].iloc[0]
            filled_count += 1
            continue
        
        
        same_title_type = df[(df['title'] == current_product['title']) & 
                           (df['product_type'] == current_product['product_type']) &
                           (df['image_src'].notnull())]
        if len(same_title_type) > 0:
            df.loc[idx, 'image_src'] = same_title_type['image_src'].iloc[0]
            filled_count += 1
            continue
            
        
        similar_in_type = df[(df['product_type'] == current_product['product_type']) &
                           (df['title'].str.contains(current_product['title'].split()[0])) &
                           (df['image_src'].notnull())]
        if len(similar_in_type) > 0:
            df.loc[idx, 'image_src'] = similar_in_type['image_src'].iloc[0]
            filled_count += 1
            continue
            
    return filled_count


initial_missing_images = df_clean['image_src'].isnull().sum()
filled_count = greedy_image_filler(df_clean)
remaining = df_clean['image_src'].isnull().sum()

print(f"\nImage filling results:")
print(f"Initial missing images: {initial_missing_images}")
print(f"Images filled by greedy algorithm: {filled_count}")
print(f"Remaining missing images: {remaining}")


print("\n\n=== FINAL DATA QUALITY CHECK ===")
print(f"Total products: {len(df_clean)}")
print(f"Missing product_type: {df_clean['product_type'].isnull().sum()}")
print(f"Missing images: {df_clean['image_src'].isnull().sum()}")
print(f"Duplicates: {df_clean.duplicated().sum()}")
print(f"Invalid prices: {(df_clean['price'] <= 0).sum()}")


print("\n=== DATA DISTRIBUTION ===")
for col in ['product_type', 'vendor']:
    print(f"\n{col}: {df_clean[col].nunique()} unique values")
    print(df_clean[col].value_counts().head())


if remaining > 0:
    stubborn_products = df_clean[df_clean['image_src'].isnull()]
    print(f"\n=== ANALYSIS OF {remaining} PRODUCTS WITHOUT IMAGES ===")
    print("Products:")
    print(stubborn_products[['title', 'product_type', 'vendor']].to_string())
    
    print("\nDistribution by product_type:")
    print(stubborn_products['product_type'].value_counts())
    
    print("\nDistribution by vendor:")
    print(stubborn_products['vendor'].value_counts())

print("\n=== DATA CLEANING COMPLETED ===")

print('\n\n EDA\n\n')

print('\n1. price statistics')
print(f"maximum price: US$ {df_clean['price'].max():.2f}")
print(f"minimum price: US$ {df_clean['price'].min():.2f}")
print(f"mean price: US$ {df_clean['price'].mean():.2f}")
most_expensive = df_clean[df_clean['price'] == df_clean['price'].max()]
print(f"name of the most expensive item: {most_expensive['title'].values[0]}")
print(f"price: U$ {most_expensive['price'].values[0]}")
print(f"category: {most_expensive['product_type'].values[0]}")
def expensive_analysis(df, threshold=500, outlier_count=100):
    expensive_items = df[df['price'] > threshold]
    items_count = len(expensive_items)
    total_products = len(df)
    percentage = (items_count / total_products) * 100

    print('\n2. expensive products analysis')
    print(f'products above US$ {threshold}: {items_count}')
    print(f'percentage: {percentage:.2f}%')
    print(f"average price of expensive items: US$ {expensive_items['price'].mean():.2f}")

    if items_count > outlier_count:
        print('classification normal')
        print(f'reason: {items_count} products above threshold')
        status = 'legal'
    elif items_count > 0:
        print('classification: outlier/special items')
        print(f'reason: only {items_count} products above threshold')
        status = 'outlier'
    else:
        print('classification: no expensive products')
        status = 'no_expensive'
    
    if items_count > 0:
        print('distribution by category')
        category_stats = expensive_items['product_type'].value_counts()
        for category, count in category_stats.items():
            print(f' {category}: {count} products')
    return status, items_count, expensive_items
status, count, expensive_df = expensive_analysis(df_clean)

print(f'final result: {status} with {count} expensive products')
print(f'\n\ntop 5 category')

top_5 = df_clean['product_type'].value_counts().head()
for i, (type, quantity) in enumerate(top_5.items(), 1):
    print(f' {i}. {type}: {quantity} products')

=== INITIAL DATA EXPLORATION ===
Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44831 entries, 0 to 44830
Data columns (total 10 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   title               44831 non-null  object 
 1   product_type        44824 non-null  object 
 2   vendor              44831 non-null  object 
 3   tags                44831 non-null  object 
 4   handle              44831 non-null  object 
 5   variant_title       44831 non-null  object 
 6   sku                 44831 non-null  object 
 7   price               44831 non-null  float64
 8   inventory_quantity  0 non-null      float64
 9   image_src           44703 non-null  object 
dtypes: float64(2), object(8)
memory usage: 3.4+ MB

Dataset Shape: (44831, 10)

Missing Values:
title                     0
product_type              7
vendor                    0
tags                      0
handle                    0
variant_title      